## Personally identifiable information (PII) in the dataset

###  Definition of PII

Personally Identifiable Information (PII) refers to any information that can identify a specific individual, either directly or indirectly.

Under the General Data Protection Regulation (GDPR), the broader term used is “personal data”, defined as "Any information relating to an identified or identifiable natural person."

An identifiable person is someone who can be identified directly (e.g., by name or ID number) or indirectly (e.g., through a combination of attributes such as date of birth, gender, and ZIP code).

Since the NovaCred dataset is used for credit scoring decisions, it contains multiple forms of personal data that fall under GDPR protection and EU AI Act obligations.


###  PII classification
We have audited the `raw_credit_applications.json` dataset and identified several categories of Personally Identifiable Information (PII). Under GDPR, these require specific technical and organizational measures to ensure lawful processing.

#### 1 Direct Identifiers
These fields can be used to uniquely identify a natural person without any additional information:
* `full_name`: Directly identifies a person. It's the most straightforward form of PII since it points to a specific individual without needing any additional context.
* `email`: Uniquely assigned to an individual, often contains the person's name, and serves as both an identifier and a contact channel. Even without a name attached, an email address can be traced back to its owner.
* `ssn`: A permanent, unique national identifier assigned to a single person. It's among the most sensitive PII because it cannot be easily changed and is widely used for identity verification across financial, governmental, and healthcare systems. Exposure creates severe risk of identity theft.
* `ip_adress`: Identifies the device and network used during the application. It can be linked back to a specific individual through their internet service provider, which means it qualifies as personal data even if it doesn't directly contain a name.

---




#### 2 Indirect identifiers (quasi-identifiers)

Not identifying on their own, but become PII when combined with other fields:

* `date_of_birth`:  A birthdate alone is shared by many people. However, when combined with other quasi-identifiers like gender and zip code, the pool of matching individuals shrinks dramatically, often to a single person. The risk lies in combination, not isolation.

* `gender`: Same principle. Knowing someone's gender tells you very little by itself, but it significantly narrows the population when crossed with location and age data. It's also a protected characteristic under anti-discrimination legislation, adding a layer of sensitivity beyond identification.

* `zip_code`: Geographic information that narrows someone's location to a small area. Some postal codes cover very few residents, which makes re-identification much easier. It also carries indirect sensitivity because residential patterns often correlate with ethnicity and socioeconomic status.

---

#### 3 Behavioral Data with Privacy Implications

* `spending_behavior` (category + amount): Not a traditional identifier, but highly personal. Detailed spending patterns can reveal sensitive aspects of someone's life such as health conditions, religious beliefs, or political affiliations depending on the categories tracked. This raises a data minimization concern: is this level of granularity necessary for the purpose of credit decisioning?
---
#### 4 Non-PII fields

* `_id`: An internal application identifier with no meaning outside the system. It doesn't relate to the person's real-world identity.

* `annual_income`, `credit_history_months`, `debt_to_income`, `savings_balance`: These are financial attributes, not identifiers. They describe characteristics rather than pointing to a specific person. However, they become personal data the moment they are linked to any of the identifiers above, because they then describe an identified individual's financial situation.

* decision object (`loan_approved`, `interest_rate`, `approved_amount`, `rejection_reason`): Not PII on its own, but once linked to an applicant it represents the outcome of automated profiling. This makes it relevant under GDPR provisions on automated decision-making, since the decision directly affects the individual's access to credit.

## Implementing Pseudonymization

### SSN Pseudonymization Demo

The SSN (Social Security Number) field is the most sensitive direct identifier in the dataset. It is a permanent, unique national identifier that cannot be changed, making its exposure a severe identity theft risk.

To protect this field while preserving analytical utility, we apply **SHA-256 hashing**, a one-way cryptographic function that converts each SSN into a fixed-length string of characters.

**Why hashing works for this use case:**
- **Irreversible:** the original SSN cannot be recovered from the hash
- **Deterministic:** the same SSN always produces the same hash, so we can still detect duplicate records without seeing the real values
- **Fixed output:** regardless of input, the hash is always 64 characters, removing any structural clues about the original

**GDPR note:** hashed data is still considered *pseudonymized* (not *anonymized*) under GDPR Article 4(5), because re-identification remains theoretically possible. This means GDPR still applies to the hashed dataset.

### Step 1: Basic Hashing with SHA-256

We start by applying SHA-256 hashing to the SSN field. SHA-256 is a one-way cryptographic function, it converts any input into a fixed 64-character string that cannot be mathematically reversed.

In [1]:
import pandas as pd
import hashlib

df = pd.read_csv("cleaned_credit_applications.csv")

# Show original SSNs
print("=== Original SSNs ===")
print(df[["_id", "full_name", "ssn"]].head())

# Apply basic SHA-256 hashing
def hash_ssn(ssn):
    return hashlib.sha256(ssn.encode()).hexdigest()

df["ssn_hashed"] = df["ssn"].apply(hash_ssn)

print("\n=== After Basic Hashing ===")
print(df[["_id", "ssn", "ssn_hashed"]].head())

=== Original SSNs ===
       _id        full_name          ssn
0  app_200      Jerry Smith  596-64-4340
1  app_037   Brandon Walker  425-69-4784
2  app_215      Scott Moore  370-78-5178
3  app_024       Thomas Lee  194-35-1833
4  app_184  Brian Rodriguez  480-41-2475

=== After Basic Hashing ===
       _id          ssn                                         ssn_hashed
0  app_200  596-64-4340  2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...
1  app_037  425-69-4784  2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...
2  app_215  370-78-5178  db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...
3  app_024  194-35-1833  c835719be02018987096d6e49529a24b1d7e7ab35c84b1...
4  app_184  480-41-2475  41c7de40dc49185886e6ecb37346ec9eabce16087b7508...


### Step 2: The Hidden Trap: Brute-Force Reversal

The hash above looks secure, but it has a critical weakness: **hashing is deterministic**. The same input always produces the same output.

SSNs follow a known format: 9 digits, roughly one billion possible combinations. An attacker who obtains the hashed dataset can simply:

1. Generate every possible SSN from `000-00-0000` to `999-99-9999`
2. Hash each one with SHA-256
3. Match the results against the hashed column

This is called a **brute-force reversal**, and with modern hardware it takes minutes, not hours.


Let's demonstrate this:

In [3]:
# Demonstrate brute-force vulnerability
# Take the first hashed SSN from our dataset
target_hash = df["ssn_hashed"].iloc[0]
original_ssn = df["ssn"].iloc[0]

print(f"Target hash: {target_hash[:40]}...")
print(f"Original SSN (known to us, unknown to attacker): {original_ssn}")

# Simulate an attacker: hash the known SSN format and match
# In reality, an attacker would loop through all 10^9 combinations
# Here we show the principle with the actual value
attack_attempt = hashlib.sha256(original_ssn.encode()).hexdigest()

print(f"\nAttacker hashes '{original_ssn}' and gets: {attack_attempt[:40]}...")
print(f"Match found: {attack_attempt == target_hash}")
print("\nThis confirms that basic hashing is vulnerable to brute-force reversal.")

Target hash: 2caf30528c21a10e1307b27f9dbbfc312f0c00d4...
Original SSN (known to us, unknown to attacker): 596-64-4340

Attacker hashes '596-64-4340' and gets: 2caf30528c21a10e1307b27f9dbbfc312f0c00d4...
Match found: True

This confirms that basic hashing is vulnerable to brute-force reversal.


### Step 3: The Fix: Salted Hashing

To defeat brute-force attacks, we add a **salt**, a secret random string that is appended to the SSN before hashing.
```
Without salt:  hash("596-64-4340")           → a1b2c3...
With salt:     hash("x7Km9pQr" + "596-64-4340") → f8d2a1... (completely different)
```

The salt makes pre-computation impossible, even if the attacker knows all possible SSN inputs, they cannot reproduce the hashes without knowing the salt.

**Critical requirement:** The salt must be:
- **Generated securely** using a cryptographically strong random generator
- **Stored separately** from the hashed data, with independent access controls
- **Never included** in the analytical dataset

This aligns directly with GDPR Article 4(5), which requires that the additional information needed for re-identification (in this case, the salt) is **kept separately** and protected by technical and organizational measures.

In [4]:
import secrets

# Generate a cryptographically secure salt
SALT = secrets.token_hex(32)

print(f"Generated salt (store this securely, separately from the data):")
print(f"{SALT}\n")

# Apply salted hashing
def salted_hash_ssn(ssn):
    return hashlib.sha256((SALT + ssn).encode()).hexdigest()

df["ssn_salted_hash"] = df["ssn"].apply(salted_hash_ssn)

print("=== Comparison: Basic Hash vs Salted Hash ===")
print(df[["_id", "ssn", "ssn_hashed", "ssn_salted_hash"]].head())

print("\nNotice how the salted hashes are completely different from the basic hashes.")
print("An attacker without the salt cannot reproduce these values.")

Generated salt (store this securely, separately from the data):
bf590f4c2ef30d5babb2418f1583fd3feebd7828e2c1e5c53888fb463648f1ed

=== Comparison: Basic Hash vs Salted Hash ===
       _id          ssn                                         ssn_hashed  \
0  app_200  596-64-4340  2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...   
1  app_037  425-69-4784  2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...   
2  app_215  370-78-5178  db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...   
3  app_024  194-35-1833  c835719be02018987096d6e49529a24b1d7e7ab35c84b1...   
4  app_184  480-41-2475  41c7de40dc49185886e6ecb37346ec9eabce16087b7508...   

                                     ssn_salted_hash  
0  8d15601299eaf099f4d37830ea73e47c955defade570e7...  
1  7476e05c39a39cd495a5fb49141ea0e4fbd1cfe97f1457...  
2  e8df5bb0d758738867a3989644e9bb940c0ea2351ecac2...  
3  b76397c84b08ae343661ba3bac450798a470d06b26a364...  
4  ec6ab2f6a9be39e43e22664c28d6baaba652cd31b57cf8...  

Notice how the salted ha

In [5]:
# Demonstrate that brute-force no longer works without the salt
target_salted_hash = df["ssn_salted_hash"].iloc[0]
original_ssn = df["ssn"].iloc[0]

# Attacker tries the same approach: hashing the SSN without the salt
attack_without_salt = hashlib.sha256(original_ssn.encode()).hexdigest()

print(f"Target salted hash:     {target_salted_hash[:40]}...")
print(f"Attacker's attempt:     {attack_without_salt[:40]}...")
print(f"Match found: {attack_without_salt == target_salted_hash}")
print("\nWithout the salt, the attacker's hash does not match.")
print("Brute-force reversal is no longer feasible.")

Target salted hash:     8d15601299eaf099f4d37830ea73e47c955defad...
Attacker's attempt:     2caf30528c21a10e1307b27f9dbbfc312f0c00d4...
Match found: False

Without the salt, the attacker's hash does not match.
Brute-force reversal is no longer feasible.


In [6]:
# Final step: drop the original SSN and the unsalted hash from the working dataset
df = df.drop(columns=["ssn", "ssn_hashed"])

print("=== Final: Only salted hash remains in the dataset ===")
print(df[["_id", "full_name", "ssn_salted_hash"]].head())

print(f"\nDataset columns after cleanup:")
print(df.columns.tolist())

=== Final: Only salted hash remains in the dataset ===
       _id        full_name                                    ssn_salted_hash
0  app_200      Jerry Smith  8d15601299eaf099f4d37830ea73e47c955defade570e7...
1  app_037   Brandon Walker  7476e05c39a39cd495a5fb49141ea0e4fbd1cfe97f1457...
2  app_215      Scott Moore  e8df5bb0d758738867a3989644e9bb940c0ea2351ecac2...
3  app_024       Thomas Lee  b76397c84b08ae343661ba3bac450798a470d06b26a364...
4  app_184  Brian Rodriguez  ec6ab2f6a9be39e43e22664c28d6baaba652cd31b57cf8...

Dataset columns after cleanup:
['_id', 'spending_behavior', 'full_name', 'email', 'ip_address', 'gender', 'date_of_birth', 'zip_code', 'annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance', 'loan_approved', 'rejection_reason', 'interest_rate', 'approved_amount', 'total_spending', 'age', 'ssn_salted_hash']


### Summary

| Step | What we did | Why |
|---|---|---|
| **Step 1** | Applied basic SHA-256 hashing | One-way function, no lookup table needed |
| **Step 2** | Demonstrated brute-force vulnerability | SSNs have a predictable format (~10⁹ combinations), making unsalted hashes reversible in minutes |
| **Step 3** | Added a cryptographic salt before hashing | Makes pre-computation infeasible, even if the attacker knows all possible SSN inputs |

The salted hash is the version retained in the working dataset. The original SSN and the unsalted hash have both been removed.

**Key takeaway:** Pseudonymization alone is not anonymization. Even with salted hashing, this data remains personal data under GDPR because re-identification is possible if the salt is compromised. All GDPR obligations, lawful basis, data subject rights, security measures, continue to apply.

## Data Quality Audit on Raw Dataset

The following audit is performed directly on `raw_credit_applications.json`, the dataset as it exists before any cleaning or transformation.

Each audit query below maps to one of the six data quality dimensions: **Accuracy, Completeness, Consistency, Timeliness, Validity, and Uniqueness**. The query patterns follow the MongoDB aggregation pipeline approach used in the NovaCred system, translated here to Python for reproducibility.

In [1]:
import json
import pandas as pd
from collections import Counter

# Load the raw dataset exactly as it would exist in the production database
with open("raw_credit_applications.json") as f:
    raw_data = json.load(f)

print(f"Total records in raw dataset: {len(raw_data)}")
print(f"\nTop-level fields per record: {list(raw_data[0].keys())}")
print(f"\nNested structure:")
print(f"  applicant_info fields: {list(raw_data[0]['applicant_info'].keys())}")
print(f"  financials fields:     {list(raw_data[0]['financials'].keys())}")
print(f"  decision fields:       {list(raw_data[0]['decision'].keys())}")
print(f"  spending_behavior:     list of category/amount pairs")

Total records in raw dataset: 502

Top-level fields per record: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']

Nested structure:
  applicant_info fields: ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
  financials fields:     ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
  decision fields:       ['loan_approved', 'rejection_reason']
  spending_behavior:     list of category/amount pairs


---
### Audit Query 1: Find Duplicates (Uniqueness)

**Data Quality Dimension: Uniqueness**

Each person should appear once in the dataset. Duplicate records can lead to double-counting in analytics, biased model training, and incorrect credit decisions. We check both the application ID (`_id`) and the SSN, which should be unique per individual.




In [2]:
# Audit Query 1a: Find duplicate application IDs
# Each _id should be unique — duplicates indicate data entry errors or system bugs

id_counts = Counter(r["_id"] for r in raw_data)
duplicate_ids = {k: v for k, v in id_counts.items() if v > 1}

print("=== Duplicate Application IDs ===")
if duplicate_ids:
    for app_id, count in duplicate_ids.items():
        print(f"\n  {app_id}: appears {count} times")
        for r in raw_data:
            if r["_id"] == app_id:
                print(f"    → {r['applicant_info']['full_name']}, SSN: {r['applicant_info'].get('ssn', 'MISSING')}")
    print(f"\nFinding: {len(duplicate_ids)} application IDs appear more than once.")
else:
    print("  No duplicate IDs found.")

=== Duplicate Application IDs ===

  app_042: appears 2 times
    → Joseph Lopez, SSN: 652-70-5530
    → Joseph Lopez, SSN: 652-70-5530

  app_001: appears 2 times
    → Stephanie Nguyen, SSN: 427-90-1892
    → Stephanie Nguyen, SSN: MISSING

Finding: 2 application IDs appear more than once.


In [3]:
# Audit Query 1b: Find duplicate SSNs
# Same SSN on different records = same person applied twice OR data entry error OR fraud

ssn_records = {}
for r in raw_data:
    ssn = r["applicant_info"].get("ssn")
    if ssn:
        ssn_records.setdefault(ssn, []).append({
            "_id": r["_id"],
            "name": r["applicant_info"]["full_name"]
        })

duplicate_ssns = {ssn: records for ssn, records in ssn_records.items() if len(records) > 1}

print("=== Duplicate SSNs ===")
if duplicate_ssns:
    for ssn, records in duplicate_ssns.items():
        print(f"\n  SSN {ssn}:")
        for rec in records:
            print(f"    → {rec['_id']}: {rec['name']}")
    
    print(f"\nFinding: {len(duplicate_ssns)} SSNs appear more than once.")
    print("  - 1 is an exact duplicate (same person, same ID — system bug)")
    print("  - 2 have different names sharing the same SSN — potential fraud or data entry error")
else:
    print("  No duplicate SSNs found.")

print("\nUNIQUENESS VERDICT: FAIL — duplicates exist at both the ID and SSN level.")

=== Duplicate SSNs ===

  SSN 652-70-5530:
    → app_042: Joseph Lopez
    → app_042: Joseph Lopez

  SSN 937-72-8731:
    → app_101: Sandra Smith
    → app_234: Samuel Hill

  SSN 780-24-9300:
    → app_088: Susan Martinez
    → app_016: Gary Wilson

Finding: 3 SSNs appear more than once.
  - 1 is an exact duplicate (same person, same ID — system bug)
  - 2 have different names sharing the same SSN — potential fraud or data entry error

UNIQUENESS VERDICT: FAIL — duplicates exist at both the ID and SSN level.


---
### Audit Query 2: Check Consistency

**Data Quality Dimension: Consistency**

Data should follow a uniform encoding across all records. Inconsistent representations of the same concept break aggregation, bias analysis, and model training.

In [4]:
# Audit Query 2a: Gender encoding consistency
# Expected: a controlled vocabulary with consistent values

gender_counts = Counter(r["applicant_info"].get("gender", "<MISSING>") for r in raw_data)

print("=== Gender Value Distribution ===")
for gender, count in gender_counts.most_common():
    label = f'"{gender}"' if gender else '""  (empty string)'
    if gender == "<MISSING>":
        label = "<MISSING> (field absent)"
    print(f"  {label:35s} → {count} records")

print(f"\nFinding: {len(gender_counts)} distinct values found instead of a consistent set.")
print("  'Male' vs 'M' and 'Female' vs 'F' represent the same concept with different encodings.")
print("  2 empty strings and 1 missing field add further inconsistency.")
print("\n  This breaks any aggregation or bias analysis that groups by gender.")

=== Gender Value Distribution ===
  "Male"                              → 195 records
  "Female"                            → 193 records
  "F"                                 → 58 records
  "M"                                 → 53 records
  ""  (empty string)                  → 2 records
  <MISSING> (field absent)            → 1 records

Finding: 6 distinct values found instead of a consistent set.
  'Male' vs 'M' and 'Female' vs 'F' represent the same concept with different encodings.
  2 empty strings and 1 missing field add further inconsistency.

  This breaks any aggregation or bias analysis that groups by gender.


In [ ]:
# Audit Query 2b: Date of birth format consistency
# Dates should follow a single format. Mixed formats cause parsing errors and incorrect age calculations

import re

dob_format_counts = Counter()
for r in raw_data:
    dob = r["applicant_info"].get("date_of_birth", "")
    if not dob:
        dob_format_counts["<MISSING>"] += 1
    elif re.match(r"^\d{4}-\d{2}-\d{2}$", dob):
        dob_format_counts["YYYY-MM-DD"] += 1
    elif re.match(r"^\d{2}/\d{2}/\d{4}$", dob):
        dob_format_counts["MM/DD/YYYY"] += 1
    elif re.match(r"^\d{4}/\d{2}/\d{2}$", dob):
        dob_format_counts["YYYY/MM/DD"] += 1
    else:
        dob_format_counts[f"other: {dob}"] += 1

print("=== Date of Birth Format Distribution ===")
for fmt, count in dob_format_counts.most_common():
    print(f"  {fmt:20s} → {count} records")

print(f"\nFinding: 3 different date formats exist in the dataset.")
print("  YYYY-MM-DD (340), MM/DD/YYYY (101), YYYY/MM/DD (56), plus 5 missing values.")
print("  Without standardization, age calculations will produce wrong results or errors.")

=== Date of Birth Format Distribution ===
  YYYY-MM-DD           → 340 records
  MM/DD/YYYY           → 101 records
  YYYY/MM/DD           → 56 records
  <MISSING>            → 5 records

Finding: 3 different date formats exist in the dataset.
  YYYY-MM-DD (340), MM/DD/YYYY (101), YYYY/MM/DD (56), plus 5 missing values.
  Without standardization, age calculations will produce wrong results or errors.


In [6]:
# Audit Query 2c: Field naming consistency in financials
# Check if the same concept is stored under different field names

field_variants = Counter()
for r in raw_data:
    if "annual_income" in r["financials"]:
        field_variants["annual_income"] += 1
    if "annual_salary" in r["financials"]:
        field_variants["annual_salary"] += 1

print("=== Income Field Naming ===")
for field, count in field_variants.most_common():
    print(f"  '{field}': {count} records")

# Show the records with annual_salary
print("\nRecords using 'annual_salary' instead of 'annual_income':")
for r in raw_data:
    if "annual_salary" in r["financials"]:
        print(f"  {r['_id']}: annual_salary = {r['financials']['annual_salary']}")

print(f"\nFinding: 5 records use 'annual_salary' instead of 'annual_income'.")
print("  These records have NO 'annual_income' field — they will appear as missing in any")
print("  analysis that looks for 'annual_income', and their salary data is silently lost.")

=== Income Field Naming ===
  'annual_income': 497 records
  'annual_salary': 5 records

Records using 'annual_salary' instead of 'annual_income':
  app_436: annual_salary = 45000
  app_421: annual_salary = 46000
  app_479: annual_salary = 94000
  app_463: annual_salary = 86000
  app_449: annual_salary = 75000

Finding: 5 records use 'annual_salary' instead of 'annual_income'.
  These records have NO 'annual_income' field — they will appear as missing in any
  analysis that looks for 'annual_income', and their salary data is silently lost.


In [7]:
# Audit Query 2d: Type consistency in annual_income
# Numeric fields should always be stored as numbers, not strings

type_issues = []
for r in raw_data:
    inc = r["financials"].get("annual_income")
    if isinstance(inc, str):
        type_issues.append({"_id": r["_id"], "annual_income": inc, "type": type(inc).__name__})

print("=== Type Inconsistency: annual_income stored as string ===")
for issue in type_issues:
    print(f"  {issue['_id']}: annual_income = '{issue['annual_income']}' (type: {issue['type']})")

print(f"\nFinding: {len(type_issues)} records store annual_income as a string instead of a number.")
print("  These values will fail in any numerical operation (mean, comparison, model input)")
print("  unless explicitly cast — a silent source of errors.")

print("\nCONSISTENCY VERDICT: FAIL — encoding, format, naming, and type inconsistencies found.")

=== Type Inconsistency: annual_income stored as string ===
  app_088: annual_income = '55000' (type: str)
  app_135: annual_income = '65000' (type: str)
  app_446: annual_income = '73000' (type: str)
  app_389: annual_income = '51000' (type: str)
  app_026: annual_income = '72000' (type: str)
  app_312: annual_income = '80000' (type: str)
  app_180: annual_income = '111000' (type: str)
  app_224: annual_income = '93000' (type: str)

Finding: 8 records store annual_income as a string instead of a number.
  These values will fail in any numerical operation (mean, comparison, model input)
  unless explicitly cast — a silent source of errors.

CONSISTENCY VERDICT: FAIL — encoding, format, naming, and type inconsistencies found.


---
### Audit Query 3: Find Missing Fields (Completeness)

**Data Quality Dimension: Completeness** 

Every record should contain all required fields. Missing data means incomplete credit decisions and, for governance fields, an inability to prove lawful processing under GDPR.



In [8]:
# Audit Query 3a: Missing values in critical applicant fields
# These fields are essential for identifying the applicant and making a credit decision

fields_to_check = {
    "ssn": lambda r: not r["applicant_info"].get("ssn"),
    "gender": lambda r: not r["applicant_info"].get("gender"),
    "date_of_birth": lambda r: not r["applicant_info"].get("date_of_birth"),
    "zip_code": lambda r: not r["applicant_info"].get("zip_code"),
    "ip_address": lambda r: not r["applicant_info"].get("ip_address"),
    "annual_income": lambda r: "annual_income" not in r["financials"],
}

print("=== Missing Values in Critical Fields ===")
for field, check_fn in fields_to_check.items():
    missing = sum(1 for r in raw_data if check_fn(r))
    pct = missing / len(raw_data) * 100
    status = "⚠" if missing > 0 else "✓"
    print(f"  {status} {field:25s} → {missing:3d} missing ({pct:.1f}%)")

=== Missing Values in Critical Fields ===
  ⚠ ssn                       →   5 missing (1.0%)
  ⚠ gender                    →   3 missing (0.6%)
  ⚠ date_of_birth             →   5 missing (1.0%)
  ⚠ zip_code                  →   2 missing (0.4%)
  ⚠ ip_address                →   5 missing (1.0%)
  ⚠ annual_income             →   5 missing (1.0%)


In [ ]:
# Audit Query 3b: Sparse optional fields
# Fields that exist on very few records: are they governed? documented?

optional_fields = {
    "processing_timestamp": lambda r: "processing_timestamp" in r and r["processing_timestamp"],
    "loan_purpose": lambda r: "loan_purpose" in r,
    "notes": lambda r: "notes" in r,
}

print("=== Sparse Optional Fields ===")
for field, check_fn in optional_fields.items():
    present = sum(1 for r in raw_data if check_fn(r))
    missing = len(raw_data) - present
    pct_missing = missing / len(raw_data) * 100
    print(f"  {field:25s} → {present:3d} present, {missing:3d} missing ({pct_missing:.1f}%)")

=== Sparse Optional Fields ===
  processing_timestamp      →  62 present, 440 missing (87.6%)
  loan_purpose              →  50 present, 452 missing (90.0%)
  notes                     →   2 present, 500 missing (99.6%)


In [10]:
# Audit Query 3c: Missing GDPR governance fields
# Under GDPR, certain metadata is required to prove lawful processing

gdpr_fields = {
    "consent_timestamp": "When did the applicant consent to data processing?",
    "retention_until": "When should this record be deleted?",
    "data_source": "Where was this data collected from?",
    "processing_purpose": "What is the lawful basis for processing?",
}

print("=== Missing GDPR Governance Fields ===")
for field, description in gdpr_fields.items():
    present = sum(1 for r in raw_data if field in r)
    print(f"  '{field}': {present}/{len(raw_data)} records")
    print(f"    → {description}")
    print()

print("Finding: NONE of the four governance fields exist in ANY record.")
print("  NovaCred cannot prove lawful basis for processing, cannot enforce retention")
print("  limits, cannot trace data origin, and cannot demonstrate consent.")
print("\nCOMPLETENESS VERDICT: FAIL — critical applicant data and ALL governance metadata missing.")

=== Missing GDPR Governance Fields ===
  'consent_timestamp': 0/502 records
    → When did the applicant consent to data processing?

  'retention_until': 0/502 records
    → When should this record be deleted?

  'data_source': 0/502 records
    → Where was this data collected from?

  'processing_purpose': 0/502 records
    → What is the lawful basis for processing?

Finding: NONE of the four governance fields exist in ANY record.
  NovaCred cannot prove lawful basis for processing, cannot enforce retention
  limits, cannot trace data origin, and cannot demonstrate consent.

COMPLETENESS VERDICT: FAIL — critical applicant data and ALL governance metadata missing.


---
### Audit Query 4: Check Data Validity

**Data Quality Dimension: Validity**

Even when fields are present, their values must conform to business rules. Income must be positive, debt-to-income ratios must be between 0 and 1, and credit history cannot be negative.

In [11]:
# Audit Query 4: Validate numerical fields against business rules
# Business Rules:
#   - annual_income must be > 0
#   - credit_history_months must be >= 0
#   - debt_to_income must be between 0 and 1
#   - savings_balance must be >= 0

print("=== Validity Violations ===\n")

# Zero or negative income
print("1) annual_income ≤ 0 (income must be positive):")
for r in raw_data:
    inc = r["financials"].get("annual_income")
    if isinstance(inc, (int, float)) and inc <= 0:
        print(f"   {r['_id']}: annual_income = {inc}")
count_inc = sum(1 for r in raw_data if isinstance(r["financials"].get("annual_income"), (int, float)) and r["financials"]["annual_income"] <= 0)
print(f"   → {count_inc} record(s)\n")

# Negative credit history
print("2) credit_history_months < 0 (cannot have negative history):")
for r in raw_data:
    chm = r["financials"].get("credit_history_months")
    if isinstance(chm, (int, float)) and chm < 0:
        print(f"   {r['_id']}: credit_history_months = {chm}")
count_chm = sum(1 for r in raw_data if isinstance(r["financials"].get("credit_history_months"), (int, float)) and r["financials"]["credit_history_months"] < 0)
print(f"   → {count_chm} record(s)\n")

# DTI above 1.0
print("3) debt_to_income > 1.0 (ratio cannot exceed 100%):")
for r in raw_data:
    dti = r["financials"].get("debt_to_income")
    if isinstance(dti, (int, float)) and dti > 1:
        print(f"   {r['_id']}: debt_to_income = {dti}")
count_dti = sum(1 for r in raw_data if isinstance(r["financials"].get("debt_to_income"), (int, float)) and r["financials"]["debt_to_income"] > 1)
print(f"   → {count_dti} record(s)\n")

# Negative savings
print("4) savings_balance < 0 (savings cannot be negative):")
for r in raw_data:
    sb = r["financials"].get("savings_balance")
    if isinstance(sb, (int, float)) and sb < 0:
        print(f"   {r['_id']}: savings_balance = {sb}")
count_sb = sum(1 for r in raw_data if isinstance(r["financials"].get("savings_balance"), (int, float)) and r["financials"]["savings_balance"] < 0)
print(f"   → {count_sb} record(s)")

total_violations = count_inc + count_chm + count_dti + count_sb
print(f"\nTotal validity violations: {total_violations} across {len(raw_data)} records.")
print("Each of these means a credit decision was made on data that violates basic business rules.")
print("\nVALIDITY VERDICT: FAIL — impossible values exist in the production dataset.")

=== Validity Violations ===

1) annual_income ≤ 0 (income must be positive):
   app_190: annual_income = 0
   → 1 record(s)

2) credit_history_months < 0 (cannot have negative history):
   app_043: credit_history_months = -10
   app_156: credit_history_months = -3
   → 2 record(s)

3) debt_to_income > 1.0 (ratio cannot exceed 100%):
   app_402: debt_to_income = 1.85
   → 1 record(s)

4) savings_balance < 0 (savings cannot be negative):
   app_290: savings_balance = -5000
   → 1 record(s)

Total validity violations: 5 across 502 records.
Each of these means a credit decision was made on data that violates basic business rules.

VALIDITY VERDICT: FAIL — impossible values exist in the production dataset.


---
### Audit Query 5: Detect Potential Bias

**AI Act Requirement: Fairness Testing**

Under the EU AI Act, high-risk AI systems (which includes credit scoring) must use training data that is representative and free from bias. The **80% Rule** (Disparate Impact ratio) is used as a screening tool: if one group's approval rate is less than 80% of another group's rate, there is evidence of potential discrimination that requires investigation.

In [12]:
# Audit Query 5a: Approval rate by gender (raw values, including inconsistent encoding)
# First, we show the raw picture — what the database actually returns

gender_stats_raw = {}
for r in raw_data:
    g = r["applicant_info"].get("gender", "")
    if not g:
        g = "<missing/empty>"
    if g not in gender_stats_raw:
        gender_stats_raw[g] = {"total": 0, "approved": 0}
    gender_stats_raw[g]["total"] += 1
    if r["decision"]["loan_approved"]:
        gender_stats_raw[g]["approved"] += 1

print("=== Approval Rate by Gender (Raw Encoding) ===\n")
print(f"  {'Gender':<20s} {'Approved':>10s} {'Total':>8s} {'Rate':>10s}")
print("  " + "-" * 50)
for g in sorted(gender_stats_raw, key=lambda x: gender_stats_raw[x]["total"], reverse=True):
    s = gender_stats_raw[g]
    rate = s["approved"] / s["total"]
    print(f"  {g:<20s} {s['approved']:>10d} {s['total']:>8d} {rate:>10.1%}")

print("\nNote: The inconsistent gender encoding (Male/M, Female/F) fragments the analysis.")
print("  A naive query would show 4+ groups instead of 2, masking the true disparity.")

=== Approval Rate by Gender (Raw Encoding) ===

  Gender                 Approved    Total       Rate
  --------------------------------------------------
  Male                        131      195      67.2%
  Female                      101      193      52.3%
  F                            26       58      44.8%
  M                            32       53      60.4%
  <missing/empty>               2        3      66.7%

Note: The inconsistent gender encoding (Male/M, Female/F) fragments the analysis.
  A naive query would show 4+ groups instead of 2, masking the true disparity.


In [13]:
# Audit Query 5b: Approval rate by gender (normalized) with Disparate Impact calculation
# Combine Male+M and Female+F to get the true picture

gender_combined = {"Male": {"total": 0, "approved": 0}, "Female": {"total": 0, "approved": 0}}
for r in raw_data:
    g = r["applicant_info"].get("gender", "")
    if g in ("Male", "M"):
        group = "Male"
    elif g in ("Female", "F"):
        group = "Female"
    else:
        continue  # skip missing/empty for this analysis
    gender_combined[group]["total"] += 1
    if r["decision"]["loan_approved"]:
        gender_combined[group]["approved"] += 1

print("=== Approval Rate by Gender (Normalized) ===\n")
print(f"  {'Gender':<12s} {'Approved':>10s} {'Total':>8s} {'Rate':>10s}")
print("  " + "-" * 42)
rates = {}
for g, s in gender_combined.items():
    rate = s["approved"] / s["total"]
    rates[g] = rate
    print(f"  {g:<12s} {s['approved']:>10d} {s['total']:>8d} {rate:>10.1%}")

# Disparate Impact calculation
di = rates["Female"] / rates["Male"]
print(f"\n  Disparate Impact Ratio: {di:.4f}")
print(f"  80% Rule Threshold:     0.80")
print(f"  Status:                 {'⚠ BELOW THRESHOLD — investigate for discrimination' if di < 0.80 else 'Above threshold'}")

print(f"\nFinding: Female applicants are approved at {rates['Female']:.1%} compared to {rates['Male']:.1%} for males.")
print(f"  The DI ratio of {di:.4f} is below the 0.80 threshold, indicating potential gender discrimination.")
print("\nBIAS VERDICT: FAIL — disparate impact detected. Investigation required under the EU AI Act.")

=== Approval Rate by Gender (Normalized) ===

  Gender         Approved    Total       Rate
  ------------------------------------------
  Male                163      248      65.7%
  Female              127      251      50.6%

  Disparate Impact Ratio: 0.7698
  80% Rule Threshold:     0.80
  Status:                 ⚠ BELOW THRESHOLD — investigate for discrimination

Finding: Female applicants are approved at 50.6% compared to 65.7% for males.
  The DI ratio of 0.7698 is below the 0.80 threshold, indicating potential gender discrimination.

BIAS VERDICT: FAIL — disparate impact detected. Investigation required under the EU AI Act.


---
### Audit Query 6: Assess Timeliness

**Data Quality Dimension: Timeliness**

Data must be up-to-date and timestamped to be useful. The `processing_timestamp` field should record when each application was processed, enabling retention enforcement, audit trails, and temporal analysis.

In [14]:
# Audit Query 6: Timeliness — processing_timestamp coverage

present = sum(1 for r in raw_data if r.get("processing_timestamp"))
missing = len(raw_data) - present
pct_missing = missing / len(raw_data) * 100

print("=== Processing Timestamp Coverage ===")
print(f"  Present: {present}/{len(raw_data)} ({present/len(raw_data)*100:.1f}%)")
print(f"  Missing: {missing}/{len(raw_data)} ({pct_missing:.1f}%)")

print(f"\nFinding: {pct_missing:.1f}% of records have no processing timestamp.")
print("  Without timestamps, NovaCred cannot determine:")
print("  - When a decision was made (audit trail)")
print("  - Whether records have exceeded their retention period (GDPR Art. 5(1)(e))")
print("  - How the model's behavior has changed over time (drift monitoring)")
print("\nTIMELINESS VERDICT: FAIL — timestamps are effectively absent from the dataset.")

=== Processing Timestamp Coverage ===
  Present: 62/502 (12.4%)
  Missing: 440/502 (87.6%)

Finding: 87.6% of records have no processing timestamp.
  Without timestamps, NovaCred cannot determine:
  - When a decision was made (audit trail)
  - Whether records have exceeded their retention period (GDPR Art. 5(1)(e))
  - How the model's behavior has changed over time (drift monitoring)

TIMELINESS VERDICT: FAIL — timestamps are effectively absent from the dataset.


---
### Data Quality Audit Summary

| Dimension | Audit | Verdict | Key Findings |
|---|---|---|---|
| **Uniqueness** | Duplicate IDs and SSNs | FAIL | 2 duplicate `_id` values; 3 SSNs shared across records (1 exact duplicate, 2 cross-person) |
| **Consistency** | Gender encoding, date formats, field naming, types | FAIL | 6 gender representations; 3 date formats; `annual_salary` vs `annual_income`; 8 strings in numeric fields |
| **Completeness** | Missing critical fields and governance metadata | FAIL | 5 missing SSNs, 5 missing income; 0/502 records have consent, retention, source, or purpose fields |
| **Validity** | Business rule violations | FAIL | 1 zero income, 2 negative credit history, 1 impossible DTI, 1 negative savings |
| **Bias (Fairness)** | Disparate impact by gender | FAIL | DI ratio ≈ 0.77, below the 0.80 threshold |
| **Timeliness** | Timestamp coverage | FAIL | 87.6% of records have no `processing_timestamp` |

**Overall Data Quality Assessment: The dataset fails on all six dimensions.** No credit decisions should be made on this data without remediation.

## GDPR Mapping
### The 7 Principles (Article 5)

> **Regulatory source:** Regulation (EU) 2016/679 of the European Parliament and of the Council of 27 April 2016 on the protection of natural persons with regard to the processing of personal data and on the free movement of such data (General Data Protection Regulation). *OJ L 119, 4.5.2016, pp. 1–88.* Available at: https://eur-lex.europa.eu/eli/reg/2016/679/oj

The following mapping evaluates NovaCred's compliance with each GDPR principle based on the specific issues identified in the Data Quality Audit above.

#### 1. Lawfulness, Fairness and Transparency — Art. 5(1)(a)

**Lawfulness:** The dataset contains no record of the legal basis under which personal data was collected and processed. The audit confirmed that `consent_timestamp`, `processing_purpose`, and `data_source` are absent from all 502 records. Under Article 6 of the GDPR (Regulation (EU) 2016/679, Art. 6), every processing activity must rely on one of six lawful bases (e.g., consent, contractual necessity, legitimate interest). NovaCred cannot currently demonstrate which basis applies to any applicant's data.

**Fairness:** The bias audit (Query 5) reveals disparate impact in loan approval rates by gender. Female applicants are approved at ~50.6% compared to ~65.7% for males, producing a Disparate Impact ratio of ~0.77, below the 0.80 threshold commonly used as a fairness benchmark. Decisions that systematically disadvantage protected groups undermine the fairness requirement, especially in automated decision-making contexts.

**Transparency:** Applicants who are rejected receive the reason `algorithm_risk_score`, which provides no meaningful explanation of how the decision was reached. Under GDPR, individuals have the right to understand how their data is being used and how decisions about them are made.

---

#### 2. Purpose Limitation — Art. 5(1)(b)

Personal data must be collected for specified, explicit and legitimate purposes. The dataset contains no documentation of the stated purpose for collecting each field. For instance:
- Is the `ip_address` collected for fraud detection, geolocation, or security logging? Without a declared purpose, its collection cannot be justified.
- Granular `spending_behavior` categories (e.g., Alcohol, Healthcare, Fitness, Gambling, Adult Entertainment) go beyond what is needed for a credit assessment. If the stated purpose is creditworthiness evaluation, collecting categories that reveal health habits, religious practices, or lifestyle choices exceeds that purpose.
- The `loan_purpose` field exists for only 50 out of 502 records (10%), with no documentation of why it is collected or why it is absent for 90% of applicants (Regulation (EU) 2016/679, Art. 5(1)(b)).

---

#### 3. Data Minimization — Art. 5(1)(c)

Only data that is adequate, relevant and necessary for the stated purpose should be collected. Several fields raise concerns:
- **`ip_address`:** Not required for credit decisioning. A credit application does not need to know the applicant's network address.
- **`spending_behavior` categories:** The audit revealed 15 distinct spending categories including `Adult Entertainment`, `Gambling`, `Alcohol`, and `Healthcare`. These can reveal sensitive information about personal habits and health conditions that is not necessary for evaluating creditworthiness. Aggregate spending totals would suffice.
- **`ssn`:** While needed for identity verification, it should not be stored in plain text after verification is complete. The principle requires minimizing not just what is collected, but how it is retained.

---

#### 4. Accuracy — Art. 5(1)(d)

Data must be accurate and kept up to date. The Data Quality Audit identified multiple accuracy issues:
- **Inconsistent gender encoding:** 6 different representations (Male, Female, M, F, empty string, missing) for a binary/ternary concept.
- **Multiple date formats:** 3 different date formats (YYYY-MM-DD, MM/DD/YYYY, YYYY/MM/DD) plus 5 missing values.
- **Field naming inconsistency:** 5 records use `annual_salary` instead of `annual_income`, silently losing income data.
- **Type inconsistency:** 8 records store `annual_income` as a string instead of a number.
- **Invalid numerical values:** 1 record with zero income, 2 with negative credit history months, 1 with debt-to-income above 1.0, and 1 with negative savings balance.

Each of these means automated credit decisions are being made on inaccurate or incomplete data.

---

#### 5. Storage Limitation — Art. 5(1)(e)

Personal data must be kept only for as long as necessary for the purpose it was collected. The dataset has:
- **No retention policy:** No `retention_until`, `created_at`, or `expires_at` field exists in any record.
- **Effectively no timestamps:** `processing_timestamp` is present in only 62 out of 502 records (12.4%), making it useless for determining how long data has been stored.
- **No deletion mechanism:** There is no evidence of a process for periodically reviewing and purging records that are no longer needed.

---

#### 6. Integrity and Confidentiality — Art. 5(1)(f)

Data must be protected against unauthorized access, loss or damage. The raw dataset stores highly sensitive identifiers in plain text:
- **`ssn`:** Social Security Numbers are fully visible. A single breach would expose permanent, irreplaceable national identifiers.
- **`full_name`, `email`:** Stored without any encryption or pseudonymization.
- **`ip_address`:** Stored in plain text, linkable to individuals through ISP records.
- **No access controls documented:** There is no indication of who can access this data, under what conditions, or with what level of authorization.

---

#### 7. Accountability — Art. 5(2)

The data controller must be able to demonstrate compliance with all of the above principles. NovaCred currently lacks:
- **No audit trail:** There is no log of who accessed the data, when decisions were made, or what factors influenced each outcome.
- **No consent records:** All 502 records lack `consent_timestamp` or `consent_type`, NovaCred cannot prove that any applicant agreed to processing.
- **No documentation of processing activities:** Article 30 of the GDPR requires a Record of Processing Activities (ROPA) (Regulation (EU) 2016/679, Art. 30). There is no evidence one exists.
- **No Data Protection Impact Assessment (DPIA):** Given that the system involves automated decision-making with legal effects on individuals (credit decisions), a DPIA is mandatory under Article 35 of the GDPR (Regulation (EU) 2016/679, Art. 35). There is no indication one has been conducted.


### Data Subject Rights at Risk

#### Right to Erasure (Right to be Forgotten) — Art. 17

Under Article 17 of the GDPR (Regulation (EU) 2016/679, Art. 17), data subjects can request the deletion of their personal data when it is no longer necessary for the original purpose, when consent is withdrawn, or when processing is unlawful.

- **No retention timestamps:** Without `created_at` or `retention_until`, NovaCred cannot determine which records have exceeded their necessary retention period.
- **No deletion mechanism:** Applicant information is spread across identity fields, financial fields, behavioral data, and decision fields. A complete erasure requires touching every part of the record.
- **Pseudonymization complicates erasure:** If SSNs are hashed (as demonstrated in the Pseudonymization section of this notebook), NovaCred needs a secure mapping to identify which hashed record belongs to a requesting individual.

---

#### Rights Related to Automated Decision-Making — Art. 22

Under Article 22 of the GDPR (Regulation (EU) 2016/679, Art. 22), data subjects have the right not to be subject to decisions based solely on automated processing that produce legal effects or significantly affect them.

- **`algorithm_risk_score` as rejection reason:** 170 out of 210 rejected applicants (81%) received this as their only explanation. The remaining 40 received marginally more specific reasons (`insufficient_credit_history`, `high_dti_ratio`, `low_income`) but still no individualized explanation.
- **No human oversight:** The dataset contains no `reviewed_by`, `reviewer_decision`, or `human_override` field. All decisions appear to be fully automated.
- **No explainability:** The `decision` object records the outcome but not the reasoning. No feature importance, risk factor breakdown, or decision log exists.
- **Legal effect is clear:** A credit rejection directly impacts a person's ability to access financial services.

## EU AI Act Classification

> **Regulatory source:** Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence (Artificial Intelligence Act). *OJ L 2024/1689, 12.7.2024.* Available at: https://eur-lex.europa.eu/eli/reg/2024/1689/oj

### NovaCred as a High-Risk AI System

Under the EU AI Act's risk-based framework (Regulation (EU) 2024/1689), AI systems used for **credit scoring and creditworthiness assessment** are explicitly classified as **High Risk** (Regulation (EU) 2024/1689, Annex III, Section 5(b)). This means NovaCred's lending model is subject to the strictest set of requirements before it can be deployed or continue operating.

### Requirements and Findings

**Risk Management System — Art. 9**

High-risk AI systems must have a continuous risk management process (Regulation (EU) 2024/1689, Art. 9) that identifies and mitigates risks throughout the system lifecycle. The Data Quality Audit found no evidence of:
- A documented risk assessment for the lending model
- Identification of known risks (bias, data quality issues, lack of explainability)
- Mitigation measures for any of these risks
- Monitoring processes to detect when the model's behavior drifts or degrades

The specific risks that should have been identified include:
- **Gender bias:** DI ratio ≈ 0.77, with female applicants approved at ~50.6% versus ~65.7% for males.
- **Data quality failures:** 6 gender encodings, 3 date formats, field naming inconsistencies, type mismatches, and invalid numerical values, all feeding directly into the model.
- **Duplicate records:** 3 duplicate SSNs and 2 duplicate application IDs contaminating the training data.

---

**Data Quality and Governance — Art. 10**

Training, validation and testing datasets for high-risk systems must be relevant, representative, free of errors, and complete (Regulation (EU) 2024/1689, Art. 10). The audit demonstrated failures on every count:
- **Not free of errors:** Inconsistent gender coding (6 representations), 3 date formats, 8 strings in numeric fields, 5 records using wrong field name (`annual_salary`), and 5 validity violations (zero income, negative credit history, impossible DTI, negative savings).
- **Not representative:** The DI ratio of ≈0.77 indicates systematic disadvantage for female applicants, suggesting the training data reflects discriminatory patterns.
- **Not complete:** 5 records missing SSN, 5 missing income data (plus 5 more with the wrong field name), 5 missing date of birth, and effectively no timestamps (87.6% missing).
- **No data governance:** Zero governance fields exist. No consent tracking, no retention policy, no processing purpose, no data source documentation.

---

**Transparency and Information — Art. 13**

High-risk systems must be designed to allow users to interpret the system's output (Regulation (EU) 2024/1689, Art. 13):
- **`algorithm_risk_score`** is the rejection reason for 81% of rejected applicants (170 out of 210), providing no interpretable information.
- No feature importance or factor breakdown is available for any decision.
- No technical documentation exists describing the model's capabilities, limitations, or known biases.

---

**Human Oversight — Art. 14**

High-risk systems must allow effective human oversight, including the ability to understand outputs, override decisions, and intervene (Regulation (EU) 2024/1689, Art. 14):
- **No human review fields:** No `reviewed_by`, `override_decision`, or `human_approval` column exists in any record.
- **No intervention mechanism:** No process exists for a human to review and override automated rejections.
- **Full automation:** The combination of `algorithm_risk_score` rejections and the complete absence of any human oversight fields indicates fully automated credit decisions.

---

**Record-Keeping — Art. 12**

High-risk systems must automatically log events to ensure traceability (Regulation (EU) 2024/1689, Art. 12):
- **Effectively no timestamps:** `processing_timestamp` is missing for 440 out of 502 records (87.6%).
- **No version tracking** for the model that produced each decision.
- **No audit log** connecting inputs to outputs.
- **No change history** showing whether any record was modified after the initial decision.

## Governance Policy Recommendations

The following recommendations are grounded in the specific issues identified across the Data Quality Audit, GDPR mapping, and EU AI Act assessment conducted in this notebook. Each recommendation references the audit finding that motivates it and the regulatory articles it addresses.

---

### 1. Pseudonymize All Direct Identifiers at Rest

**Audit Finding:** The raw dataset stores `ssn`, `full_name`, `email`, and `ip_address` in plain text. SSNs are permanent national identifiers, a single breach would cause irreversible harm to all 502 applicants.

**Recommendation:** Apply salted SHA-256 hashing to the `ssn` field immediately, as demonstrated in the Pseudonymization section of this notebook. Extend pseudonymization to `full_name` and `email` in any analytical or shared copy of the dataset. The salt must be stored separately from the hashed data, with independent access permissions. Original identifiers should only be accessible in a restricted production environment with role-based access controls.

**Addresses:** Regulation (EU) 2016/679, Art. 5(1)(f) — Integrity and Confidentiality; Regulation (EU) 2016/679, Art. 25 — Data Protection by Design.

---

### 2. Remove or Justify Collection of `ip_address` and Granular `spending_behavior` Categories

**Audit Finding:** `ip_address` serves no documented purpose in credit decisioning. `spending_behavior` records 15 granular categories including Adult Entertainment, Gambling, Alcohol, and Healthcare, which can reveal sensitive information qualifying as special category data under Article 9 of the GDPR (Regulation (EU) 2016/679, Art. 9).

**Recommendation:** Remove `ip_address` from the dataset entirely unless a specific, lawful purpose is documented. Replace granular spending categories with an aggregate total spending value, which preserves analytical utility without exposing sensitive lifestyle information. If specific categories are retained, document the legal basis under Article 9 of the GDPR (Regulation (EU) 2016/679, Art. 9) for processing each one.

**Addresses:** Regulation (EU) 2016/679, Art. 5(1)(b) — Purpose Limitation; Regulation (EU) 2016/679, Art. 5(1)(c) — Data Minimization; Regulation (EU) 2016/679, Art. 9 — Special Categories of Data.

---

### 3. Implement Bias Monitoring with Defined Thresholds

**Audit Finding:** The Disparate Impact ratio for gender is ≈0.77, below the 0.80 threshold. Female applicants are approved at ~50.6% versus ~65.7% for males. The inconsistent gender encoding (6 different values) also complicates any fairness analysis.

**Recommendation:** Establish a bias monitoring framework that computes Disparate Impact ratios by gender, age group, and their intersection on a regular schedule (e.g., monthly or per model retrain). Set a minimum DI threshold of 0.80. Any metric below this must trigger a mandatory human review of the model and its training data. Document all monitoring results as part of the risk management obligations under the EU AI Act (Regulation (EU) 2024/1689, Art. 9).

**Addresses:** Regulation (EU) 2016/679, Art. 5(1)(a) — Fairness; Regulation (EU) 2024/1689, Art. 9 — Risk Management; Regulation (EU) 2024/1689, Art. 10 — Data Quality.

---

### 4. Replace `algorithm_risk_score` with Explainable Rejection Reasons

**Audit Finding:** 170 out of 210 rejected applicants (81%) receive `algorithm_risk_score` as their only rejection reason. The remaining 40 receive slightly more specific reasons, but none include an individualized explanation of which factors drove the decision.

**Recommendation:** Replace the generic `algorithm_risk_score` label with a structured explanation that includes the top contributing factors and their direction (e.g., "debt_to_income above threshold," "credit_history_months below minimum"). Store these in a new `rejection_factors` field. This does not require exposing the full model, it requires a human-readable summary of key decision drivers for each case.

**Addresses:** Regulation (EU) 2016/679, Art. 22 — Automated Decision-Making; Regulation (EU) 2024/1689, Art. 13 — Transparency.

---

### 5. Introduce Human Review for Automated Rejections

**Audit Finding:** Zero records contain any indication of human involvement in decisions, no `reviewed_by`, `override_decision`, or `human_approval` field exists. All 502 decisions appear to be fully automated.

**Recommendation:** Implement mandatory human-in-the-loop review for all loan rejections before the decision is communicated to the applicant. Add three fields to the schema: `reviewed_by` (identifier of the human reviewer), `review_timestamp`, and `override_decision` (whether the reviewer agreed with, modified, or reversed the automated outcome). A random sample of approvals should also be reviewed to detect false positives and model drift.

**Addresses:** Regulation (EU) 2016/679, Art. 22 — Automated Decision-Making; Regulation (EU) 2024/1689, Art. 14 — Human Oversight.

---

### 6. Implement a Data Retention Policy with Timestamps

**Audit Finding:** No `created_at`, `retention_until`, or `expires_at` field exists in any record. `processing_timestamp` is missing for 87.6% of records (440 out of 502).

**Recommendation:** Add mandatory `created_at` and `retention_until` fields to every record at the point of collection. Define retention periods by purpose: approved applications retained for the loan duration plus a regulatory holding period; rejected applications retained for a maximum of 12 months to allow disputes. Implement an automated process that flags and deletes records past their retention date. Backfill `processing_timestamp` for existing records where possible, and mark the 440 records with missing timestamps for priority review.

**Addresses:** Regulation (EU) 2016/679, Art. 5(1)(e) — Storage Limitation; Regulation (EU) 2024/1689, Art. 12 — Record-Keeping.

---

### 7. Establish an Audit Trail and Record of Processing Activities

**Audit Finding:** The dataset contains no access logs, no version tracking, no change history, and no consent records. NovaCred cannot trace any decision, respond to a data subject access request within one month, or demonstrate compliance to a regulator.

**Recommendation:** Implement audit logging that records every access, modification, and decision event with a timestamp, user identifier, and action description. Create a Record of Processing Activities (ROPA) as required by Article 30 of the GDPR (Regulation (EU) 2016/679, Art. 30). Add `consent_timestamp` and `consent_type` fields to the schema. Conduct a Data Protection Impact Assessment (DPIA) as required by Article 35 of the GDPR (Regulation (EU) 2016/679, Art. 35) for systems involving automated decision-making with legal effects.

**Addresses:** Regulation (EU) 2016/679, Art. 5(2) — Accountability; Regulation (EU) 2016/679, Art. 30 — Records of Processing Activities; Regulation (EU) 2016/679, Art. 35 — Data Protection Impact Assessment; Regulation (EU) 2024/1689, Art. 12 — Record-Keeping.

---

### 8. Standardize Data Quality Controls at Ingestion

**Audit Finding:** The audit found 6 gender representations, 3 date formats, 5 records using `annual_salary` instead of `annual_income`, 8 records with strings in numeric fields, and 5 records with invalid numerical values. All of these entered the dataset because there are no validation rules at ingestion.

**Recommendation:** Implement input validation schemas that enforce data quality at the point of collection:
- Restrict `gender` to a controlled vocabulary (`Male`, `Female`, `Other`, `Prefer not to say`)
- Enforce a single date format (ISO 8601: `YYYY-MM-DD`)
- Standardize the income field name to `annual_income` and reject alternatives
- Enforce numeric types for all financial fields
- Reject `credit_history_months` and `savings_balance` below 0; reject `debt_to_income` above 1.0; require `annual_income` > 0
- Make `date_of_birth`, `annual_income`, `gender`, and `ssn` mandatory
- Log all rejected submissions for review rather than silently accepting invalid data

**Addresses:** Regulation (EU) 2016/679, Art. 5(1)(d) — Accuracy; Regulation (EU) 2024/1689, Art. 10 — Data and Data Governance.